In [1]:
import numpy as np
from utils import load_dataset , one_hot
import tensorflow as tf

SEED=1137
np.random.seed(SEED)
tf.random.set_seed(SEED)

dropout_rate=0.5

In [2]:
X_train,Y_train,X_test,Y_test=load_dataset()

Y_one_hot_train=Y_train
print(Y_one_hot_train.shape)#1,m
Y_one_hot_train=tf.reshape(Y_one_hot_train,[-1])
print(Y_one_hot_train.shape)#m,
Y_one_hot_train=tf.one_hot(Y_one_hot_train,depth=6,axis=0)
print(Y_one_hot_train.shape)#6,m

print(Y_train[0,3])
print(Y_one_hot_train[:][3])

print(X_train.shape)

#for Y_test
Y_one_hot_test=Y_test
Y_one_hot_test=tf.reshape(Y_one_hot_test,[-1])
Y_one_hot_test=tf.one_hot(Y_one_hot_test,depth=6,axis=0)

X_train = X_train.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0


print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Image dtype:", X_train.dtype)


(1, 1080)
(1080,)
(6, 1080)
5
tf.Tensor([0. 0. 0. ... 0. 0. 0.], shape=(1080,), dtype=float32)
(1080, 64, 64, 3)
X_train shape: (1080, 64, 64, 3)
X_test shape: (120, 64, 64, 3)
Image dtype: float32


Parameters

In [3]:
def get_parameters(filters1, filters2, fc_units):
    parameters = {}
    initializer = tf.keras.initializers.HeNormal()

    parameters["W1"] = tf.Variable(
        initializer([3,3,3,filters1]), name="W1"
    )

    parameters["W2"] = tf.Variable(
        initializer([3,3,filters1,filters2]), name="W2"
    )

    parameters["b1"] = tf.Variable(
        tf.zeros([filters1]), name="b1"
    )

    parameters["b2"] = tf.Variable(
        tf.zeros([filters2]), name="b2"
    )

    parameters["Wfc1"] = tf.Variable(
        initializer([fc_units, 16*16*filters2]),
        name="Wfc1"
    )

    parameters["Wfc2"] = tf.Variable(
        initializer([6,fc_units]),
        name="Wfc2"
    )

    parameters["bfc1"] = tf.Variable(
        tf.zeros([fc_units]), name="bfc1"
    )

    parameters["bfc2"] = tf.Variable(
        tf.zeros([6]), name="bfc2"
    )

    return parameters

forward_propagation

In [4]:
def forward_propagation(X,parameters,training=False, dropout_rate=0.0):

    W1=parameters["W1"]
    W2=parameters["W2"]
    b1=parameters["b1"]
    b2=parameters["b2"]
    Wfc1=parameters["Wfc1"]
    Wfc2=parameters["Wfc2"]
    bfc1=parameters["bfc1"]
    bfc2=parameters["bfc2"]


    Z1=tf.nn.conv2d(X,W1,strides=[1,1,1,1],padding='SAME')
    A1=tf.nn.relu(Z1+b1)

    P1=tf.nn.max_pool2d(A1,ksize=[1,2,2,1],strides=[1,2,2,1],padding='SAME')

    Z2=tf.nn.conv2d(P1,W2,strides=[1,1,1,1],padding='SAME')
    A2=tf.nn.relu(Z2+b2)

    P2=tf.nn.max_pool2d(A2,ksize=[1,2,2,1],strides=[1,2,2,1],padding='SAME')
    flatten_layer = tf.keras.layers.Flatten()
    A_fc0=flatten_layer(P2)
    A_fc0=tf.transpose(A_fc0)

    Z_fc1=tf.add((tf.matmul(Wfc1,A_fc0)),tf.reshape(bfc1,(-1,1)))
    A_fc1=tf.nn.relu(Z_fc1)

    if training:
        A_fc1 = tf.nn.dropout(
            A_fc1,
            rate=dropout_rate
        )

    Z_fc2=tf.add((tf.matmul(Wfc2,A_fc1)),tf.reshape(bfc2,(-1,1)))



    return Z_fc2 #shape=(6,1080)



# parameters=get_parameters()
# ZL=forward_propagation(X_train,parameters)
# print(ZL.shape)

Cost

In [5]:
def compute_cost(ZL , Y , parameters):
    m=ZL.shape[1]
    cost_per_image=tf.nn.softmax_cross_entropy_with_logits(logits=tf.transpose(ZL),labels=tf.transpose(Y))
    cost=tf.reduce_mean(cost_per_image)

    lambd1=0.05
    lambd2=0
    #L2
    l2_cost=(tf.nn.l2_loss(parameters["W1"])+
             tf.nn.l2_loss(parameters["W2"])+
             tf.nn.l2_loss(parameters["b1"])+
             tf.nn.l2_loss(parameters["b2"])+
             tf.nn.l2_loss(parameters["Wfc1"])+
             tf.nn.l2_loss(parameters["Wfc2"])+
             tf.nn.l2_loss(parameters["bfc1"])+
             tf.nn.l2_loss(parameters["bfc2"]))

    #L1
    l1_cost = (
    tf.reduce_sum(tf.abs(parameters["W1"])) +
    tf.reduce_sum(tf.abs(parameters["W2"])) +
    tf.reduce_sum(tf.abs(parameters["b1"])) +
    tf.reduce_sum(tf.abs(parameters["b2"])) +
    tf.reduce_sum(tf.abs(parameters["Wfc1"])) +
    tf.reduce_sum(tf.abs(parameters["Wfc2"])) +
    tf.reduce_sum(tf.abs(parameters["bfc1"])) +
    tf.reduce_sum(tf.abs(parameters["bfc2"]))
)

    return cost+ ( (lambd1/m) * l1_cost ) + ((lambd2/m)*l2_cost)

model


In [6]:
def model(X,Y,learning_rate,no_iteration,dropout_rate):
    #small
    # parameters=get_parameters(16,32,64)
    #medium
    parameters=get_parameters(32,64,128)
    #large
    # parameters=get_parameters(64,128,256)
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate)

    for i in range (no_iteration):
        with tf.GradientTape() as tape:
            ZL=forward_propagation(X,parameters,training=False,dropout_rate=dropout_rate)
            cost=compute_cost(ZL=ZL,Y=Y,parameters=parameters)
 
        grads=tape.gradient(cost,list(parameters.values()))
        optimizer.apply_gradients(zip(grads,parameters.values()))
        if i%10==0:
            print("after",i,"iteration cost = ",cost)

    return parameters       




prediction

In [7]:
def prediction(X,parameters):
    
    ZL=forward_propagation(X,parameters,training=False,dropout_rate=0.0)
    # prob=tf.nn.softmax(ZL)
    pred=tf.argmax(ZL,axis=0)
    
    return pred


def accuracy(X,Y,parameters):
    pred=prediction(X,parameters)
    
    # 1,m to m,
    labels=tf.cast(tf.reshape(Y,[-1]),tf.int64)

    correct_pred=tf.equal(pred,labels)

    accuracy=tf.reduce_mean(tf.cast(correct_pred,"float"))
    return accuracy



    

testing the model

In [8]:
#getting optimized paras
parameters=model(X_train,Y_one_hot_train,learning_rate=.001,no_iteration=1000,dropout_rate=dropout_rate)

after 0 iteration cost =  tf.Tensor(119.27515, shape=(), dtype=float32)
after 10 iteration cost =  tf.Tensor(74.41234, shape=(), dtype=float32)
after 20 iteration cost =  tf.Tensor(30.140186, shape=(), dtype=float32)
after 30 iteration cost =  tf.Tensor(18.570583, shape=(), dtype=float32)
after 40 iteration cost =  tf.Tensor(12.00485, shape=(), dtype=float32)
after 50 iteration cost =  tf.Tensor(10.20784, shape=(), dtype=float32)
after 60 iteration cost =  tf.Tensor(8.968537, shape=(), dtype=float32)
after 70 iteration cost =  tf.Tensor(8.279354, shape=(), dtype=float32)
after 80 iteration cost =  tf.Tensor(7.7216616, shape=(), dtype=float32)
after 90 iteration cost =  tf.Tensor(7.2937717, shape=(), dtype=float32)
after 100 iteration cost =  tf.Tensor(6.938129, shape=(), dtype=float32)
after 110 iteration cost =  tf.Tensor(6.6320934, shape=(), dtype=float32)
after 120 iteration cost =  tf.Tensor(6.380321, shape=(), dtype=float32)
after 130 iteration cost =  tf.Tensor(6.1699767, shape=(

In [9]:
#using para to get prediction

acc_train=accuracy(X_train,Y_train,parameters)
acc_test=accuracy(X_test,Y_test,parameters)

print("train_accuracy=",acc_train)
print("test_accuracy=",acc_test)

train_accuracy= tf.Tensor(1.0, shape=(), dtype=float32)
test_accuracy= tf.Tensor(0.90833336, shape=(), dtype=float32)


In [10]:
Z_test = forward_propagation(
    X_test,
    parameters,
    training=False,
    dropout_rate=0.0
)

probs = tf.nn.softmax(Z_test, axis=0)

print(tf.argmax(Z_test[:, :10], axis=0))
print(tf.transpose(probs[:, :5]))

tf.Tensor([0 0 0 5 1 0 3 1 5 1], shape=(10,), dtype=int64)
tf.Tensor(
[[9.9980348e-01 1.9628851e-04 2.0746623e-07 1.8006810e-08 4.0962167e-09
  1.3070719e-12]
 [9.6328229e-01 5.6786737e-03 3.0993432e-02 4.4042339e-05 1.1916554e-06
  4.5814693e-07]
 [9.9996591e-01 1.0780143e-12 1.8837531e-05 6.5876311e-06 8.5783977e-06
  3.3079452e-08]
 [3.4090544e-08 3.7563579e-09 5.9993585e-07 4.6738269e-06 2.0894452e-08
  9.9999475e-01]
 [3.5219203e-04 9.9639964e-01 3.0371000e-03 2.1106917e-04 2.2493175e-08
  4.1945079e-09]], shape=(5, 6), dtype=float32)
